# All-binding dataset anatomy audit — DevSet1010 / TestSet300 / TestSetNew46

This notebook is a **pre-model mechanistic audit** for the three residue-level tasks:

- `metal_binding`
- `nuclear_binding`
- `small_binding`

`any_ligand_binding_sites` is retained only as an overall reference.

The audit answers two questions before specialized model design:

## A. Are the binding classes biologically/statistically different?
For each class we profile:

- residue prevalence;
- positive-protein fraction;
- site burden per positive protein;
- amino-acid enrichment;
- local sequence-context enrichment;
- positive clustering;
- pairwise label overlap;
- class-specific signatures.

## B. Are those signatures stable across Dev1010, Test300 and TestNew46?
We quantify:

- prevalence shift;
- protein-length shift;
- whole-protein AA-composition shift;
- positive-residue AA-signature shift;
- cross-split correlation of enrichment fingerprints;
- optional cached-feature mechanism fingerprints.

A feature family should only be promoted into a task-specific model if its signal is:
1. mechanistically interpretable; and
2. reasonably stable across the three benchmark splits.

No predictor is trained in the core audit.


In [ ]:
# ============================================================
# 0. DRIVE / DEPENDENCIES / PATH AUTO-DETECTION
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import ast, json, math, os, subprocess, sys
from collections import Counter

import numpy as np
import pandas as pd

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "scipy", "scikit-learn", "biopython"],
    check=True
)
from scipy.stats import fisher_exact
from scipy.spatial.distance import jensenshannon

CANDIDATE_BASES = [
    Path('/content/drive/MyDrive/Protein-binding-Phi'),
    Path('/content/drive/MyDrive/Protein-binding'),
]

BASE = None
for b in CANDIDATE_BASES:
    if (
        (b/'data/development_set/full_grouped_train_binding_sites_df.csv').exists()
        and (b/'data/development_set/full_grouped_test_binding_sites_df.csv').exists()
        and (b/'data/independent_set/grouped_test_46_new_binding_sites.csv').exists()
    ):
        BASE = b
        break

if BASE is None:
    raise FileNotFoundError('Project CSVs not found.')

OUT = BASE/'results'/'ALL_BINDINGS_DATASET_ANATOMY_V1'
OUT.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = BASE/'data/development_set/full_grouped_train_binding_sites_df.csv'
TEST300_CSV = BASE/'data/development_set/full_grouped_test_binding_sites_df.csv'
TEST46_CSV = BASE/'data/independent_set/grouped_test_46_new_binding_sites.csv'

TASKS = ['metal_binding','nuclear_binding','small_binding']
ALL_LABELS = TASKS + ['any_ligand_binding_sites']

print('BASE =', BASE)
print('OUT  =', OUT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE = /content/drive/MyDrive/Protein-binding-Phi
OUT  = /content/drive/MyDrive/Protein-binding-Phi/results/ALL_BINDINGS_DATASET_ANATOMY_V1


In [ ]:
# ============================================================
# 1. LOAD DATA + EXACT OFFICIAL DevSet1010
# ============================================================
def parse_list(v):
    if isinstance(v, str):
        return list(ast.literal_eval(v))
    return list(v)

def load_df(path):
    df = pd.read_csv(path)
    df['prot_id'] = df['prot_id'].astype(str)
    df['sequence'] = df['sequence'].astype(str)

    for c in ALL_LABELS:
        if c in df.columns:
            df[c] = df[c].apply(parse_list)

    if not df.prot_id.is_unique:
        raise RuntimeError(f'{path}: duplicate prot_id')

    for r in df.itertuples():
        for c in ALL_LABELS:
            if hasattr(r,c):
                y = getattr(r,c)
                if len(y) != len(r.sequence):
                    raise RuntimeError(
                        f'{path}/{r.prot_id}/{c}: {len(y)} != {len(r.sequence)}'
                    )
    return df

raw_train = load_df(TRAIN_CSV)
test300 = load_df(TEST300_CSV)
test46 = load_df(TEST46_CSV)

other = (
    Path('/content/drive/MyDrive/Protein-binding')
    if BASE.name == 'Protein-binding-Phi'
    else Path('/content/drive/MyDrive/Protein-binding-Phi')
)

BN24_CANDIDATES = [
    BASE/'baseline_reproduction_master_v6'/'repos'/'bindnode24',
    BASE/'repos'/'bindnode24',
    other/'baseline_reproduction_master_v6'/'repos'/'bindnode24',
    other/'repos'/'bindnode24',
]

BN24 = next((p for p in BN24_CANDIDATES if (p/'.git').exists()), None)

if BN24 is None:
    BN24 = Path('/content/bindnode24_dataset_anatomy')
    if not BN24.exists():
        subprocess.run(
            ['git','clone','-q','https://github.com/erckert/bindNode24.git',str(BN24)],
            check=True
        )

fold_root = BN24/'dataset'/'cv_splits'/'predicted_structures'
OFFICIAL_FOLDS = {}
official_order = []
seen = set()

for k in range(1,6):
    fp = fold_root/f'ids_split{k}.txt'
    ids = [x.strip() for x in fp.read_text().splitlines() if x.strip()]
    overlap = seen.intersection(ids)
    if overlap:
        raise RuntimeError(f'Fold overlap: {sorted(overlap)[:10]}')
    OFFICIAL_FOLDS[k] = ids
    official_order.extend(ids)
    seen.update(ids)

official = set(official_order)
raw_ids = set(raw_train.prot_id)

missing = official - raw_ids
if missing:
    raise RuntimeError(f'Official Dev IDs missing from project CSV: {sorted(missing)[:20]}')

train_lookup = raw_train.set_index('prot_id',drop=False)
dev = train_lookup.loc[official_order].reset_index(drop=True)

assert len(dev) == 1010
assert len(test300) == 300
assert len(test46) == 46

SETS = {
    'DevSet1010':dev,
    'TestSet300':test300,
    'TestSetNew46':test46,
}

print('Raw dev lineage:',len(raw_train))
print('Exact DevSet1010:',len(dev))
print('Excluded IDs:',sorted(raw_ids-official))
print('Fold sizes:',{k:len(v) for k,v in OFFICIAL_FOLDS.items()})


Raw dev lineage: 1014
Exact DevSet1010: 1010
Excluded IDs: ['C8BD48', 'P84801', 'Q00277', 'Q9NZV6']
Fold sizes: {1: 202, 2: 203, 3: 203, 4: 202, 5: 200}


## 2. Core task anatomy

This table is the first sanity check.

For each task and split it reports:
- residue prevalence;
- number/fraction of positive proteins;
- positive-site burden;
- protein length.

If a class is concentrated in very few proteins, a residue classifier can accidentally learn protein-level context rather than true site localization.


In [ ]:
# ============================================================
# 2. CORE PREVALENCE / PROTEIN BURDEN
# ============================================================
rows=[]

for dataset,df in SETS.items():
    for task in TASKS:
        y_all = np.concatenate([
            np.asarray(x,dtype=np.int8)
            for x in df[task]
        ])

        per_protein = np.asarray([
            int(np.sum(x))
            for x in df[task]
        ])

        pos_protein = per_protein > 0

        rows.append({
            'dataset':dataset,
            'task':task,
            'n_proteins':len(df),
            'n_residues':len(y_all),
            'n_positive_residues':int(y_all.sum()),
            'residue_prevalence':float(y_all.mean()),
            'n_positive_proteins':int(pos_protein.sum()),
            'positive_protein_fraction':float(pos_protein.mean()),
            'median_positive_residues_per_positive_protein':
                float(np.median(per_protein[pos_protein])) if pos_protein.any() else 0,
            'mean_positive_residues_per_positive_protein':
                float(np.mean(per_protein[pos_protein])) if pos_protein.any() else 0,
            'max_positive_residues_one_protein':int(per_protein.max()),
            'median_protein_length':float(np.median(df.sequence.str.len())),
        })

core = pd.DataFrame(rows)
display(core)

pivot_prev = core.pivot(
    index='task',
    columns='dataset',
    values='residue_prevalence'
)
print('Residue prevalence by task/split:')
display(pivot_prev)

core.to_csv(OUT/'core_task_anatomy.csv',index=False)


,dataset,task,n_proteins,n_residues,n_positive_residues,residue_prevalence,n_positive_proteins,positive_protein_fraction,median_positive_residues_per_positive_protein,mean_positive_residues_per_positive_protein,max_positive_residues_one_protein,median_protein_length
0,DevSet1010,metal_binding,1010,170181,2370,0.013926,454,0.449505,4.0,5.220264,34,155.0
1,DevSet1010,nuclear_binding,1010,170181,2689,0.015801,108,0.106931,19.5,24.898148,112,155.0
2,DevSet1010,small_binding,1010,170181,9236,0.054272,603,0.597030,13.0,15.316750,98,155.0
3,TestSet300,metal_binding,300,62689,881,0.014054,124,0.413333,4.0,7.104839,69,184.5
4,TestSet300,nuclear_binding,300,62689,1470,0.023449,66,0.220000,20.0,22.272727,64,184.5
5,TestSet300,small_binding,300,62689,3906,0.062308,220,0.733333,16.0,17.754545,118,184.5
6,TestSetNew46,metal_binding,46,6227,77,0.012366,15,0.326087,3.0,5.133333,24,126.0
7,TestSetNew46,nuclear_binding,46,6227,77,0.012366,10,0.217391,6.5,7.700000,18,126.0
8,TestSetNew46,small_binding,46,6227,425,0.068251,25,0.543478,11.0,17.000000,96,126.0


Residue prevalence by task/split:


dataset,DevSet1010,TestSet300,TestSetNew46
task,,,
metal_binding,0.013926,0.014054,0.012366
nuclear_binding,0.015801,0.023449,0.012366
small_binding,0.054272,0.062308,0.068251


## 3. Pairwise label overlap

We profile overlap both at:
- residue level;
- protein level.

This tells us whether the three tasks represent genuinely distinct site phenotypes or largely overlapping generic-binding regions.


In [ ]:
# ============================================================
# 3. PAIRWISE LABEL OVERLAP
# ============================================================
pairs = [
    ('metal_binding','nuclear_binding'),
    ('metal_binding','small_binding'),
    ('nuclear_binding','small_binding'),
]

rows=[]

for dataset,df in SETS.items():
    for a,b in pairs:
        ya = np.concatenate([np.asarray(x,dtype=bool) for x in df[a]])
        yb = np.concatenate([np.asarray(x,dtype=bool) for x in df[b]])

        pa = np.array([np.sum(x)>0 for x in df[a]],dtype=bool)
        pb = np.array([np.sum(x)>0 for x in df[b]],dtype=bool)

        inter = int((ya & yb).sum())
        union = int((ya | yb).sum())

        rows.append({
            'dataset':dataset,
            'task_a':a,
            'task_b':b,
            'residue_intersection':inter,
            'residue_jaccard':inter/max(union,1),
            'fraction_a_positive_residues_also_b':
                inter/max(int(ya.sum()),1),
            'fraction_b_positive_residues_also_a':
                inter/max(int(yb.sum()),1),
            'protein_intersection':int((pa & pb).sum()),
            'protein_jaccard':
                int((pa & pb).sum())/max(int((pa | pb).sum()),1),
        })

overlap = pd.DataFrame(rows)
display(overlap)
overlap.to_csv(OUT/'pairwise_label_overlap.csv',index=False)


,dataset,task_a,task_b,residue_intersection,residue_jaccard,fraction_a_positive_residues_also_b,fraction_b_positive_residues_also_a,protein_intersection,protein_jaccard
0,DevSet1010,metal_binding,nuclear_binding,52,0.010385,0.021941,0.019338,22,0.040741
1,DevSet1010,metal_binding,small_binding,289,0.025537,0.121941,0.031291,131,0.141469
2,DevSet1010,nuclear_binding,small_binding,4,0.000336,0.001488,0.000433,5,0.007082
3,TestSet300,metal_binding,nuclear_binding,65,0.028434,0.073780,0.044218,27,0.165644
4,TestSet300,metal_binding,small_binding,236,0.051857,0.267877,0.060420,66,0.237410
5,TestSet300,nuclear_binding,small_binding,111,0.021083,0.075510,0.028418,33,0.130435
6,TestSetNew46,metal_binding,nuclear_binding,0,0.000000,0.000000,0.000000,0,0.000000
7,TestSetNew46,metal_binding,small_binding,3,0.006012,0.038961,0.007059,3,0.081081
8,TestSetNew46,nuclear_binding,small_binding,1,0.001996,0.012987,0.002353,1,0.029412


## 4. Amino-acid enrichment fingerprint for each binding type

For every task/split, each amino acid receives:
- positive frequency;
- global frequency;
- log2 enrichment;
- Fisher odds ratio.

This provides a compact chemical fingerprint of each binding class.


In [ ]:
# ============================================================
# 4. AA ENRICHMENT FINGERPRINT
# ============================================================
AA20 = list("ACDEFGHIKLMNPQRSTVWY")
rows=[]

for dataset,df in SETS.items():
    for task in TASKS:
        pos=[]
        neg=[]
        allseq=[]

        for r in df.itertuples():
            seq = r.sequence
            y = np.asarray(getattr(r,task),dtype=np.int8)

            allseq.extend(seq)
            pos.extend([aa for aa,yy in zip(seq,y) if yy])
            neg.extend([aa for aa,yy in zip(seq,y) if not yy])

        ac = Counter(allseq)
        pc = Counter(pos)
        nc = Counter(neg)

        nall = sum(ac.values())
        npos = sum(pc.values())
        nneg = sum(nc.values())

        for aa in AA20:
            a = pc[aa]
            b = npos-a
            c = nc[aa]
            d = nneg-c

            odds,p = fisher_exact([[a,b],[c,d]])

            fa = ac[aa]/max(nall,1)
            fp = a/max(npos,1)

            rows.append({
                'dataset':dataset,
                'task':task,
                'aa':aa,
                'global_frequency':fa,
                'positive_frequency':fp,
                'log2_enrichment':float(
                    np.log2((fp+1e-8)/(fa+1e-8))
                ),
                'odds_ratio':float(odds),
                'fisher_p':float(p),
                'positive_count':int(a),
            })

aa = pd.DataFrame(rows)
aa.to_csv(OUT/'aa_enrichment_all_tasks.csv',index=False)

for task in TASKS:
    print('\n###',task)
    p = aa[aa.task.eq(task)].pivot(
        index='aa',
        columns='dataset',
        values='log2_enrichment'
    )
    p['mean'] = p.mean(axis=1)
    display(p.sort_values('mean',ascending=False))



### metal_binding


dataset,DevSet1010,TestSet300,TestSetNew46,mean
aa,,,,
C,3.251116,2.943472,3.052132,3.082240
H,2.860037,2.731703,3.637095,3.076278
D,1.625745,1.197473,1.789098,1.537439
E,1.170030,0.941808,1.145915,1.085917
N,-0.310294,-0.012774,-0.668089,-0.330386
S,-0.965005,-0.272194,-0.557283,-0.598161
T,-0.843927,-0.684927,-0.390385,-0.639746
Q,-0.925531,-0.989631,-1.186026,-1.033729
K,-1.218076,-0.674200,-2.280850,-1.391042



### nuclear_binding


dataset,DevSet1010,TestSet300,TestSetNew46,mean
aa,,,,
R,1.673145,1.696532,2.389422,1.919700
K,1.214918,0.952195,1.526504,1.231206
H,0.536904,0.693540,0.052133,0.427526
S,0.346351,0.460509,0.179683,0.328848
T,0.146071,0.383826,0.024652,0.184850
Y,0.354500,0.320715,-0.313517,0.120566
Q,0.299944,0.228698,-0.186027,0.114205
G,0.272437,0.122654,-0.250430,0.048220
N,0.507523,0.066786,-0.668089,-0.031260



### small_binding


dataset,DevSet1010,TestSet300,TestSetNew46,mean
aa,,,,
W,1.336660,1.131470,2.288039,1.585389
C,1.158674,1.051334,1.462069,1.224026
H,1.099739,0.905153,0.979917,0.994936
Y,0.881483,0.845386,0.543879,0.756916
F,0.770698,0.507942,0.932381,0.737007
M,0.287149,0.117254,0.703077,0.369160
R,0.292289,0.605339,-0.276745,0.206961
G,-0.092724,0.213676,-0.036891,0.028021
L,-0.139730,-0.173318,0.300887,-0.004054


## 5. Binding-class distinctiveness

We compare amino-acid enrichment fingerprints between tasks.

If metal/nuclear/small have weakly correlated fingerprints, separate feature ontologies are justified.

If two tasks are nearly identical, a shared representation/head may be more reasonable.


In [ ]:
# ============================================================
# 5. TASK-SIGNATURE SIMILARITY
# ============================================================
task_signature_rows=[]

for dataset in SETS:
    p = aa[aa.dataset.eq(dataset)].pivot(
        index='aa',
        columns='task',
        values='log2_enrichment'
    )

    corr = p.corr()

    print('\n',dataset,'AA-enrichment signature correlation')
    display(corr)

    for a,b in pairs:
        # pair names already match task names
        task_signature_rows.append({
            'dataset':dataset,
            'task_a':a,
            'task_b':b,
            'AA_enrichment_correlation':
                float(corr.loc[a,b]),
        })

task_signature_similarity = pd.DataFrame(task_signature_rows)
task_signature_similarity.to_csv(
    OUT/'task_signature_similarity.csv',
    index=False
)



 DevSet1010 AA-enrichment signature correlation


task,metal_binding,nuclear_binding,small_binding
task,,,
metal_binding,1.000000,0.023635,0.329567
nuclear_binding,0.023635,1.000000,0.064959
small_binding,0.329567,0.064959,1.000000



 TestSet300 AA-enrichment signature correlation


task,metal_binding,nuclear_binding,small_binding
task,,,
metal_binding,1.000000,0.003646,0.445814
nuclear_binding,0.003646,1.000000,0.311117
small_binding,0.445814,0.311117,1.000000



 TestSetNew46 AA-enrichment signature correlation


task,metal_binding,nuclear_binding,small_binding
task,,,
metal_binding,1.000000,-0.128079,-0.362438
nuclear_binding,-0.128079,1.000000,-0.444932
small_binding,-0.362438,-0.444932,1.000000


## 6. Local sequence-context fingerprints

We measure local fractions around each residue for interpretable chemical groups.

This is not intended to replace ESM.  
It asks whether explicit context statistics show stable task-specific mechanisms.


In [ ]:
# ============================================================
# 6. LOCAL SEQUENCE-CONTEXT FINGERPRINTS
# ============================================================
GROUPS = {
    'basic_KRH':set('KRH'),
    'strong_basic_KR':set('KR'),
    'acidic_DE':set('DE'),
    'polar_NQSTY':set('NQSTY'),
    'aromatic_FWYH':set('FWYH'),
    'hydrophobic_AVILMFWY':set('AVILMFWY'),
    'sulfur_CM':set('CM'),
    'gly_pro_GP':set('GP'),
}

RADII = [2,4,7,10]

def frac(seq,i,chars,r):
    lo=max(0,i-r)
    hi=min(len(seq),i+r+1)
    w=seq[lo:hi]
    return sum(x in chars for x in w)/max(len(w),1)

rows=[]

for dataset,df in SETS.items():
    for task in TASKS:
        posvals={(g,r):[] for g in GROUPS for r in RADII}
        negvals={(g,r):[] for g in GROUPS for r in RADII}

        for rec in df.itertuples():
            y=np.asarray(getattr(rec,task),dtype=np.int8)

            for i,yy in enumerate(y):
                target = posvals if yy else negvals

                for g,chars in GROUPS.items():
                    for rad in RADII:
                        target[(g,rad)].append(
                            frac(rec.sequence,i,chars,rad)
                        )

        for g in GROUPS:
            for rad in RADII:
                p=np.asarray(posvals[(g,rad)],dtype=float)
                n=np.asarray(negvals[(g,rad)],dtype=float)

                sp=np.sqrt((p.var()+n.var())/2+1e-12)
                smd=(p.mean()-n.mean())/sp

                rows.append({
                    'dataset':dataset,
                    'task':task,
                    'feature':g,
                    'radius':rad,
                    'positive_mean':float(p.mean()),
                    'negative_mean':float(n.mean()),
                    'standardized_mean_difference':float(smd),
                })

seqctx = pd.DataFrame(rows)
seqctx.to_csv(OUT/'sequence_context_all_tasks.csv',index=False)

for dataset in SETS:
    print('\n###',dataset,'top task/context effects')
    display(
        seqctx[seqctx.dataset.eq(dataset)]
        .assign(abs_smd=lambda x:x.standardized_mean_difference.abs())
        .sort_values('abs_smd',ascending=False)
        .head(30)
    )



### DevSet1010 top task/context effects


,dataset,task,feature,radius,positive_mean,negative_mean,standardized_mean_difference,abs_smd
39,DevSet1010,nuclear_binding,strong_basic_KR,10,0.193859,0.111398,0.904444,0.904444
35,DevSet1010,nuclear_binding,basic_KRH,10,0.219909,0.135779,0.865650,0.865650
38,DevSet1010,nuclear_binding,strong_basic_KR,7,0.200565,0.111473,0.851918,0.851918
34,DevSet1010,nuclear_binding,basic_KRH,7,0.226693,0.135829,0.817477,0.817477
37,DevSet1010,nuclear_binding,strong_basic_KR,4,0.209679,0.111538,0.759237,0.759237
33,DevSet1010,nuclear_binding,basic_KRH,4,0.237092,0.135868,0.737445,0.737445
36,DevSet1010,nuclear_binding,strong_basic_KR,2,0.215346,0.111590,0.612885,0.612885
32,DevSet1010,nuclear_binding,basic_KRH,2,0.243907,0.135901,0.602672,0.602672
55,DevSet1010,nuclear_binding,hydrophobic_AVILMFWY,10,0.344062,0.410199,-0.593539,0.593539
54,DevSet1010,nuclear_binding,hydrophobic_AVILMFWY,7,0.338220,0.410193,-0.568127,0.568127



### TestSet300 top task/context effects


,dataset,task,feature,radius,positive_mean,negative_mean,standardized_mean_difference,abs_smd
135,TestSet300,nuclear_binding,strong_basic_KR,10,0.175937,0.114201,0.745482,0.745482
131,TestSet300,nuclear_binding,basic_KRH,10,0.205377,0.141529,0.739560,0.739560
130,TestSet300,nuclear_binding,basic_KRH,7,0.213333,0.141369,0.704509,0.704509
134,TestSet300,nuclear_binding,strong_basic_KR,7,0.182442,0.114067,0.702020,0.702020
129,TestSet300,nuclear_binding,basic_KRH,4,0.224278,0.141257,0.629106,0.629106
133,TestSet300,nuclear_binding,strong_basic_KR,4,0.191811,0.113984,0.623622,0.623622
128,TestSet300,nuclear_binding,basic_KRH,2,0.237211,0.141091,0.550365,0.550365
132,TestSet300,nuclear_binding,strong_basic_KR,2,0.203469,0.113880,0.546982,0.546982
138,TestSet300,nuclear_binding,acidic_DE,7,0.086430,0.125159,-0.465495,0.465495
137,TestSet300,nuclear_binding,acidic_DE,4,0.076686,0.125358,-0.465114,0.465114



### TestSetNew46 top task/context effects


,dataset,task,feature,radius,positive_mean,negative_mean,standardized_mean_difference,abs_smd
231,TestSetNew46,nuclear_binding,strong_basic_KR,10,0.225371,0.119163,1.284928,1.284928
230,TestSetNew46,nuclear_binding,strong_basic_KR,7,0.236305,0.118749,1.206456,1.206456
229,TestSetNew46,nuclear_binding,strong_basic_KR,4,0.264935,0.118448,1.171317,1.171317
227,TestSetNew46,nuclear_binding,basic_KRH,10,0.242476,0.144449,1.115311,1.115311
226,TestSetNew46,nuclear_binding,basic_KRH,7,0.255822,0.143999,1.042229,1.042229
225,TestSetNew46,nuclear_binding,basic_KRH,4,0.284416,0.143584,1.029340,1.029340
228,TestSetNew46,nuclear_binding,strong_basic_KR,2,0.274892,0.118347,0.937934,0.937934
224,TestSetNew46,nuclear_binding,basic_KRH,2,0.293074,0.143480,0.834840,0.834840
251,TestSetNew46,nuclear_binding,sulfur_CM,10,0.018190,0.046194,-0.655366,0.655366
250,TestSetNew46,nuclear_binding,sulfur_CM,7,0.017454,0.046305,-0.597664,0.597664


## 7. Positive-site clustering by task

This tests whether positives tend to form local runs along the sequence.

It is only a sequence-level proxy for spatial patch organization, but it helps distinguish:
- isolated coordination residues;
- broader interface patches.


In [ ]:
# ============================================================
# 7. POSITIVE CLUSTERING
# ============================================================
def clusters(pos,max_gap):
    pos=np.asarray(pos,dtype=int)
    if len(pos)==0:
        return []

    out=[[int(pos[0])]]
    for x in pos[1:]:
        if x-out[-1][-1] <= max_gap:
            out[-1].append(int(x))
        else:
            out.append([int(x)])
    return out

rows=[]

for dataset,df in SETS.items():
    for task in TASKS:
        for rec in df.itertuples():
            y=np.asarray(getattr(rec,task),dtype=np.int8)
            pos=np.flatnonzero(y)

            for gap in [1,2,3,5]:
                cc=clusters(pos,gap)

                rows.append({
                    'dataset':dataset,
                    'task':task,
                    'prot_id':str(rec.prot_id),
                    'max_gap':gap,
                    'n_positive':len(pos),
                    'n_clusters':len(cc),
                    'largest_cluster':max((len(c) for c in cc),default=0),
                    'fraction_positive_in_cluster_ge2':
                        sum(len(c) for c in cc if len(c)>=2)/max(len(pos),1),
                })

cluster = pd.DataFrame(rows)

cluster_summary = (
    cluster[cluster.n_positive.gt(0)]
    .groupby(['dataset','task','max_gap'])
    .agg(
        positive_proteins=('prot_id','nunique'),
        mean_largest_cluster=('largest_cluster','mean'),
        mean_fraction_positive_in_cluster_ge2=(
            'fraction_positive_in_cluster_ge2','mean'
        )
    )
    .reset_index()
)

display(cluster_summary)
cluster.to_csv(OUT/'positive_cluster_per_protein_all_tasks.csv',index=False)
cluster_summary.to_csv(OUT/'positive_cluster_summary_all_tasks.csv',index=False)


,dataset,task,max_gap,positive_proteins,mean_largest_cluster,mean_fraction_positive_in_cluster_ge2
0,DevSet1010,metal_binding,1,454,1.541850,0.209377
1,DevSet1010,metal_binding,2,454,1.995595,0.402746
2,DevSet1010,metal_binding,3,454,2.396476,0.606434
3,DevSet1010,metal_binding,5,454,2.722467,0.732779
4,DevSet1010,nuclear_binding,1,108,5.907407,0.715371
5,DevSet1010,nuclear_binding,2,108,8.379630,0.838169
6,DevSet1010,nuclear_binding,3,108,10.416667,0.914436
7,DevSet1010,nuclear_binding,5,108,12.296296,0.958507
8,DevSet1010,small_binding,1,603,3.311774,0.540334
9,DevSet1010,small_binding,2,603,4.167496,0.691326


## 8. Cross-split stability of each binding mechanism

For every task, we compare Dev1010 with Test300 and Test46 using:

- whole-protein amino-acid composition JS distance;
- positive-residue amino-acid composition JS distance;
- correlation of positive AA-enrichment fingerprints;
- prevalence ratio.

This directly answers whether a mechanism discovered in Dev has a chance to generalize.


In [ ]:
# ============================================================
# 8. CROSS-SPLIT STABILITY
# ============================================================
def aa_dist(df,task=None,positive_only=False):
    c=Counter()

    for rec in df.itertuples():
        if not positive_only:
            c.update(rec.sequence)
        else:
            y=np.asarray(getattr(rec,task),dtype=np.int8)
            c.update([
                aa for aa,yy in zip(rec.sequence,y)
                if yy
            ])

    v=np.asarray([c[a] for a in AA20],dtype=float)
    return v/v.sum() if v.sum() else np.zeros_like(v)

dev_global=aa_dist(dev)

rows=[]

for task in TASKS:
    dev_pos=aa_dist(dev,task,True)

    dev_e = (
        aa[
            aa.dataset.eq('DevSet1010')
            & aa.task.eq(task)
        ]
        .set_index('aa')
        .loc[AA20,'log2_enrichment']
        .to_numpy()
    )

    dev_prev=float(
        core[
            core.dataset.eq('DevSet1010')
            & core.task.eq(task)
        ].residue_prevalence.iloc[0]
    )

    for dataset,df in SETS.items():
        test_global=aa_dist(df)
        test_pos=aa_dist(df,task,True)

        test_e=(
            aa[
                aa.dataset.eq(dataset)
                & aa.task.eq(task)
            ]
            .set_index('aa')
            .loc[AA20,'log2_enrichment']
            .to_numpy()
        )

        prev=float(
            core[
                core.dataset.eq(dataset)
                & core.task.eq(task)
            ].residue_prevalence.iloc[0]
        )

        rows.append({
            'task':task,
            'dataset':dataset,
            'JS_global_AA_vs_Dev':
                float(jensenshannon(dev_global,test_global,base=2)),
            'JS_positive_AA_vs_Dev':
                float(jensenshannon(dev_pos,test_pos,base=2)),
            'AA_enrichment_correlation_vs_Dev':
                float(np.corrcoef(dev_e,test_e)[0,1]),
            'residue_prevalence':
                prev,
            'prevalence_ratio_vs_Dev':
                prev/max(dev_prev,1e-12),
        })

stability=pd.DataFrame(rows)
display(stability)
stability.to_csv(OUT/'cross_split_stability_all_tasks.csv',index=False)


,task,dataset,JS_global_AA_vs_Dev,JS_positive_AA_vs_Dev,AA_enrichment_correlation_vs_Dev,residue_prevalence,prevalence_ratio_vs_Dev
0,metal_binding,DevSet1010,0.000000,0.000000,1.000000,0.013926,1.000000
1,metal_binding,TestSet300,0.018990,0.127894,0.970311,0.014054,1.009130
2,metal_binding,TestSetNew46,0.044683,0.270355,0.499581,0.012366,0.887922
3,nuclear_binding,DevSet1010,0.000000,0.000000,1.000000,0.015801,1.000000
4,nuclear_binding,TestSet300,0.018990,0.082046,0.936348,0.023449,1.484042
5,nuclear_binding,TestSetNew46,0.044683,0.255069,0.338963,0.012366,0.782586
6,small_binding,DevSet1010,0.000000,0.000000,1.000000,0.054272,1.000000
7,small_binding,TestSet300,0.018990,0.060307,0.951285,0.062308,1.148069
8,small_binding,TestSetNew46,0.044683,0.096467,0.906395,0.068251,1.257585


## 9. Optional interpretable cached-feature fingerprint

If the existing `GEAR_ULTIMATE` backend/cache is available, this section loads the already prepared feature stores and computes a descriptive effect vector for each task.

For each feature block:
1. each dimension is standardized across residues in a split;
2. we compute mean(positive) − mean(negative);
3. the block score is RMS standardized separation across dimensions.

This is **not model performance**.  
It is a mechanism fingerprint showing which existing feature families differ most between positives and negatives.

Useful blocks include:
- SPATIAL / LOCAL
- ATOM / CHEM
- ACCESS
- METALSPEC
- ELEC3D
- TOPO / SOLV
- SEQMOTIF / SEQCTX

Cross-split stability of these block effects is especially informative.


In [ ]:
# ============================================================
# 9. OPTIONAL CACHED-FEATURE MECHANISM FINGERPRINT
# ============================================================
GEAR_DIR_CANDIDATES = [
    BASE/'GEAR_ULTIMATE',
    other/'GEAR_ULTIMATE',
]

GEAR_DIR = next(
    (p for p in GEAR_DIR_CANDIDATES if (p/'gear_ultimate').exists()),
    None
)

feature_fingerprint = pd.DataFrame()

if GEAR_DIR is None:
    print('GEAR_ULTIMATE backend not found -> optional feature fingerprint skipped.')
else:
    sys.path.insert(0,str(GEAR_DIR))

    from gear_ultimate import gear_suite as suite
    from gear_ultimate import gear_core as core_mod

    import torch
    DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    ctx=suite.prepare_context(
        base=BASE,
        run_version='all_bindings_anatomy_context',
        models=('MLP',),
        ensure_base=True,
        add_mech=True,
        add_multi=True,
        add_seqctx=True,
        esm_batch_size=8,
        device=DEVICE,
    )

    STORE_KEY = {
        'DevSet1010':'train',
        'TestSet300':'test300',
        'TestSetNew46':'test46',
    }

    def _store_mapping(store):
        if isinstance(store,dict):
            return store
        for attr in ('records','proteins','data','items_by_id'):
            obj=getattr(store,attr,None)
            if isinstance(obj,dict):
                return obj
        raise TypeError(type(store))

    def _container(rec):
        if isinstance(rec,dict):
            if isinstance(rec.get('features'),dict):
                return rec['features']
            return rec
        for attr in ('features','feature_dict','blocks'):
            obj=getattr(rec,attr,None)
            if isinstance(obj,dict):
                return obj
        return None

    def get_block(store,pid,b):
        mp=_store_mapping(store)
        rec=mp[str(pid)] if str(pid) in mp else mp[pid]
        cont=_container(rec)

        if isinstance(cont,dict) and b in cont:
            return np.asarray(cont[b])
        if isinstance(rec,dict) and b in rec:
            return np.asarray(rec[b])
        if hasattr(rec,b):
            return np.asarray(getattr(rec,b))
        raise KeyError(b)

    BLOCKS = [
        'SPATIAL','LOCAL','ATOM','CHEM',
        'ACCESS','SHELL2','ENV','CONF',
        'METALSPEC','ELEC3D','TOPO','SOLV',
        'SEQMOTIF','SEQCTX',
    ]

    rows=[]

    for dataset,df in SETS.items():
        store=ctx.stores[STORE_KEY[dataset]]

        for b in BLOCKS:
            Xparts=[]
            Yparts={t:[] for t in TASKS}
            ok=True

            for rec in df.itertuples():
                try:
                    x=get_block(store,str(rec.prot_id),b)
                except Exception:
                    ok=False
                    break

                if x.shape[0] != len(rec.sequence):
                    ok=False
                    break

                Xparts.append(
                    x.astype(np.float32,copy=False)
                )

                for t in TASKS:
                    Yparts[t].append(
                        np.asarray(getattr(rec,t),dtype=np.int8)
                    )

            if not ok:
                print('Skip unavailable block:',dataset,b)
                continue

            X=np.concatenate(Xparts,axis=0)

            mu=X.mean(axis=0,keepdims=True)
            sd=X.std(axis=0,keepdims=True)
            Z=(X-mu)/(sd+1e-6)

            for t in TASKS:
                y=np.concatenate(Yparts[t]).astype(bool)

                if y.sum()==0 or (~y).sum()==0:
                    continue

                effect=Z[y].mean(axis=0)-Z[~y].mean(axis=0)

                rows.append({
                    'dataset':dataset,
                    'task':t,
                    'block':b,
                    'dim':X.shape[1],
                    'effect_rms':float(np.sqrt(np.mean(effect**2))),
                    'effect_abs_mean':float(np.mean(np.abs(effect))),
                    'effect_signed_mean':float(np.mean(effect)),
                })

    feature_fingerprint=pd.DataFrame(rows)

    if len(feature_fingerprint):
        display(
            feature_fingerprint.sort_values(
                ['dataset','task','effect_rms'],
                ascending=[True,True,False]
            )
        )
        feature_fingerprint.to_csv(
            OUT/'cached_feature_mechanism_fingerprint.csv',
            index=False
        )


[DevSet1010] Reused ESM cache: DevSet1010_ESM2_t33_mean_last2_1280_fp16.pkl
[DevSet1010] Reused structure cache: DevSet1010_spatial_local_v2.pkl
[TestSet300] Reused ESM cache: TestSet300_ESM2_t33_mean_last2_1280_fp16.pkl
[TestSet300] Reused structure cache: TestSet300_spatial_local_v2.pkl
[TestSetNew46] Reused ESM cache: TestSetNew46_ESM2_t33_mean_last2_1280_fp16.pkl
[TestSetNew46] Reused structure cache: TestSetNew46_spatial_local_v2.pkl
[DevSet1010] Reused ATOM cache: DevSet1010_atomic_coordination_v1.pkl
[DevSet1010] Reused CHEM cache: DevSet1010_chemistry_v1_24d.pkl
[TestSet300] Reused ATOM cache: TestSet300_atomic_coordination_v1.pkl
[TestSet300] Reused CHEM cache: TestSet300_chemistry_v1_24d.pkl
[TestSetNew46] Reused ATOM cache: TestSetNew46_atomic_coordination_v1.pkl
[TestSetNew46] Reused CHEM cache: TestSetNew46_chemistry_v1_24d.pkl
[DevSet1010] Reused MECH1 cache: DevSet1010_mechanistic_v1.pkl
[DevSet1010] Reused MULTI2 cache: DevSet1010_multisignal_v2.pkl
[TestSet300] Reused 

,dataset,task,block,dim,effect_rms,effect_abs_mean,effect_signed_mean
6,DevSet1010,metal_binding,ATOM,15,2.803836,2.713833,1.742295
24,DevSet1010,metal_binding,METALSPEC,18,2.185424,2.126305,2.126305
30,DevSet1010,metal_binding,TOPO,12,1.454468,1.146200,1.146200
15,DevSet1010,metal_binding,SHELL2,12,1.117565,1.100767,0.732756
9,DevSet1010,metal_binding,CHEM,24,0.892474,0.693548,0.475073
...,...,...,...,...,...,...,...
110,TestSetNew46,small_binding,METALSPEC,18,0.106780,0.103168,0.103168
101,TestSetNew46,small_binding,SHELL2,12,0.090433,0.083191,0.025018
119,TestSetNew46,small_binding,SOLV,10,0.090239,0.080976,0.039770
125,TestSetNew46,small_binding,SEQCTX,182,0.089644,0.068321,0.008745


## 10. Automatic decision tables

The final tables answer:

### Task distinctiveness
Do metal, nuclear and small show different chemical/context fingerprints?

### Split stability
For each task, does its positive-residue signature remain similar from Dev → Test300/Test46?

### Candidate feature families
Which mechanisms are:
- interpretable;
- distinct for a task;
- stable enough across splits to justify explicit engineering?


In [ ]:
# ============================================================
# 10. AUTOMATIC DECISION TABLES
# ============================================================

# ----- A. Task distinctiveness from AA signature -----
task_corr_by_split=[]

for dataset in SETS:
    p=aa[aa.dataset.eq(dataset)].pivot(
        index='aa',
        columns='task',
        values='log2_enrichment'
    )
    corr=p.corr()

    vals=[]
    for i,a in enumerate(TASKS):
        for b in TASKS[i+1:]:
            vals.append(corr.loc[a,b])

    task_corr_by_split.append({
        'dataset':dataset,
        'mean_pairwise_task_AA_signature_correlation':float(np.mean(vals)),
        'max_pairwise_task_AA_signature_correlation':float(np.max(vals)),
    })

task_distinctiveness=pd.DataFrame(task_corr_by_split)
display(task_distinctiveness)

# ----- B. Stability verdict -----
stability_view=stability.copy()

def stability_flag(row):
    corr=row['AA_enrichment_correlation_vs_Dev']
    js=row['JS_positive_AA_vs_Dev']
    pr=row['prevalence_ratio_vs_Dev']

    if row['dataset']=='DevSet1010':
        return 'REFERENCE'

    if corr >= 0.75 and js <= 0.25 and 0.5 <= pr <= 2.0:
        return 'STABLE'
    if corr >= 0.5 and js <= 0.4:
        return 'MODERATE_SHIFT'
    return 'STRONG_SHIFT'

stability_view['stability_flag']=stability_view.apply(stability_flag,axis=1)
display(stability_view)

# ----- C. Core mechanism hints from AA/context only -----
hints=[]

for task in TASKS:
    aa_p=(
        aa[aa.task.eq(task)]
        .pivot(index='aa',columns='dataset',values='log2_enrichment')
    )
    aa_p['min']=aa_p.min(axis=1)
    consistent=aa_p[aa_p['min']>0].sort_values('min',ascending=False)

    ctx=(
        seqctx[seqctx.task.eq(task)]
        .groupby(['feature','radius'])
        .standardized_mean_difference
        .agg(['mean','min','max'])
        .reset_index()
    )

    hints.append({
        'task':task,
        'consistently_enriched_AA':
            ','.join(consistent.index.tolist()[:8]),
        'strongest_context_feature':
            str(ctx.iloc[ctx['mean'].abs().argmax()]['feature'])
            if len(ctx) else '',
        'strongest_context_mean_SMD':
            float(ctx.iloc[ctx['mean'].abs().argmax()]['mean'])
            if len(ctx) else np.nan,
    })

mechanism_hints=pd.DataFrame(hints)
display(mechanism_hints)

task_distinctiveness.to_csv(OUT/'task_distinctiveness.csv',index=False)
stability_view.to_csv(OUT/'stability_verdicts.csv',index=False)
mechanism_hints.to_csv(OUT/'mechanism_hints.csv',index=False)

print('\nSaved to:',OUT)


,dataset,mean_pairwise_task_AA_signature_correlation,max_pairwise_task_AA_signature_correlation
0,DevSet1010,0.139387,0.329567
1,TestSet300,0.253526,0.445814
2,TestSetNew46,-0.311816,-0.128079


,task,dataset,JS_global_AA_vs_Dev,JS_positive_AA_vs_Dev,AA_enrichment_correlation_vs_Dev,residue_prevalence,prevalence_ratio_vs_Dev,stability_flag
0,metal_binding,DevSet1010,0.000000,0.000000,1.000000,0.013926,1.000000,REFERENCE
1,metal_binding,TestSet300,0.018990,0.127894,0.970311,0.014054,1.009130,STABLE
2,metal_binding,TestSetNew46,0.044683,0.270355,0.499581,0.012366,0.887922,STRONG_SHIFT
3,nuclear_binding,DevSet1010,0.000000,0.000000,1.000000,0.015801,1.000000,REFERENCE
4,nuclear_binding,TestSet300,0.018990,0.082046,0.936348,0.023449,1.484042,STABLE
5,nuclear_binding,TestSetNew46,0.044683,0.255069,0.338963,0.012366,0.782586,STRONG_SHIFT
6,small_binding,DevSet1010,0.000000,0.000000,1.000000,0.054272,1.000000,REFERENCE
7,small_binding,TestSet300,0.018990,0.060307,0.951285,0.062308,1.148069,STABLE
8,small_binding,TestSetNew46,0.044683,0.096467,0.906395,0.068251,1.257585,STABLE


,task,consistently_enriched_AA,strongest_context_feature,strongest_context_mean_SMD
0,metal_binding,"C,H,D,E",hydrophobic_AVILMFWY,-0.403268
1,nuclear_binding,"R,K,S,H,T",strong_basic_KR,0.978285
2,small_binding,"W,C,H,Y,F,M",acidic_DE,-0.326610



Saved to: /content/drive/MyDrive/Protein-binding-Phi/results/ALL_BINDINGS_DATASET_ANATOMY_V1


# How to use this audit

Do **not** choose features simply because they are biologically plausible.

For each task, prioritize feature families that satisfy all three:

```text
Task specificity
      +
mechanistic meaning
      +
cross-split stability
```

Example interpretation:

- Metal: strong H/C/D/E chemistry + ATOM/METALSPEC signal → coordination-site reasoning.
- Nuclear: stable K/R/H + accessibility/electrostatic/patch signal → surface electrostatic recognition.
- Small molecule: broader hydrophobic/aromatic/pocket-like signature → pocket/shape/chemistry reasoning.

If TestNew46 shows a strong shift for a task, treat Test46 as a genuine external robustness test rather than expecting identical performance to Test300.
